# LLM-Guided Embedding Adaptation — Demo & Test

This notebook demonstrates and tests `tritopic.adaptation` — a ClusterLLM-style
module that adapts TriTopic's embedder to a small budget of LLM triplet
judgments on *your own* corpus, closing the gap between a generic off-the-shelf
embedder and one tuned for your domain.

**References**
- **ClusterLLM** (Zhang, Wang & Shang, EMNLP 2023) — the triplet-query fine-tuning approach this module implements.
- **PRISM** (Douglas, Balci & Aylett-Bullock, WWW 2026) — sampling/evaluation ideas for sparse LLM supervision.
- **Viswanathan et al., "Large Language Models Enable Few-Shot Clustering"** (TACL 2024) — the cheaper keyphrase-expansion and low-confidence-correction extras.

**What this notebook covers, in order:**

1. Load a real corpus (20 Newsgroups subset) and fit a baseline TriTopic model.
2. **Test the pipeline for free** — an *oracle* labeler that answers triplet judgments from ground truth (no API key, no cost) proves the adaptation plumbing itself works, with inline assertions.
3. Compare baseline vs. adapted embeddings on the same TriTopic config (`compare_embedders`).
4. Swap in a **real LLM** (Claude / GPT / Gemini) — one line change.
5. The two cheaper extras: LLM keyphrase expansion and post-hoc low-confidence correction.

Cells 1–3 run with **no API key and no network beyond the cached 20 Newsgroups
data** — safe to run top-to-bottom to confirm the feature works in your
environment before spending any LLM budget on it.

---
## 1. Setup and data

A small 20 Newsgroups subset (4 categories, ~600 docs) — big enough to be a
real test, small enough to fit and adapt in well under a minute on a CPU-only
machine. We use LSA (TF-IDF → SVD) embeddings as the baseline: fast,
deterministic, no model download, and exactly what `tritopic.cumulative`'s own
benchmarks use for CI. Swap in a real sentence-transformers model any time
(see the commented cell below) — the pipeline works identically either way.

In [1]:
# Uncomment on a fresh environment (e.g. Kaggle/Colab).
# %pip install -q "tritopic[full] @ git+https://github.com/nevil-mathew/topic-extraction-poc.git@batch-clustering"

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

categories = ["sci.med", "sci.space", "rec.autos", "comp.graphics"]

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
)

rng = np.random.default_rng(0)
idx = rng.choice(len(newsgroups.data), size=min(600, len(newsgroups.data)), replace=False)
documents = [newsgroups.data[i].strip() or "empty" for i in idx]
true_labels = newsgroups.target[idx]

# Filter out very short documents (near-empty after header/footer removal)
mask = np.array([len(doc) >= 50 for doc in documents])
documents = [doc for doc, keep in zip(documents, mask) if keep]
true_labels = true_labels[mask]

print(f"{len(documents)} documents across {len(categories)} categories")
for i, name in enumerate(categories):
    print(f"  {name:20s}: {(true_labels == i).sum()} docs")

566 documents across 4 categories
  sci.med             : 159 docs
  sci.space           : 136 docs
  rec.autos           : 129 docs
  comp.graphics       : 142 docs


In [2]:
from tritopic.cumulative.datasets import lsa_embed

baseline_embeddings = lsa_embed(documents, dim=64)
print("baseline_embeddings shape:", baseline_embeddings.shape)

# --- Optional: use a real embedding model instead (needs network + a model download) ---
# from sentence_transformers import SentenceTransformer
# encoder = SentenceTransformer("all-MiniLM-L6-v2")
# baseline_embeddings = encoder.encode(documents, normalize_embeddings=True, show_progress_bar=True)

baseline_embeddings shape: (566, 64)


## 2. Fit the baseline model

Standard `TriTopic.fit()` with the precomputed embeddings from above.

In [3]:
from tritopic import TriTopic, TriTopicConfig

baseline_config = TriTopicConfig(
    use_dim_reduction=False,
    use_iterative_refinement=False,
    n_consensus_runs=4,
    min_cluster_size=5,
    n_neighbors=15,
    random_state=42,
    verbose=False,
)

model = TriTopic(config=baseline_config)
model.fit(documents, embeddings=baseline_embeddings)

n_topics = len([t for t in model.topics_ if t.topic_id != -1])
print(f"Baseline: {n_topics} topics, {np.mean(model.labels_ == -1):.1%} outliers")
model.get_topic_info()[lambda d: d.Topic != -1].sort_values("Size", ascending=False).head(10)

Baseline: 9 topics, 0.0% outliers


,Topic,Size,Keywords,All_Keywords,Keyword_Scores,Label,Description,Representative_Docs,Coherence
0,0,102,"hiv, aids, cancer, health, disease","[hiv, aids, cancer, health, disease, medical, ...","[0.029582438341461213, 0.020284562123049632, 0...",Topic 0,None,"[393, 511, 83, 62, 53]",None
1,1,90,"edu, tmp, files, ftp, _the","[edu, tmp, files, ftp, _the, cs, com, books, k...","[0.03500360155170332, 0.02724865908573648, 0.0...",Topic 1,None,"[481, 308, 321, 173, 300]",None
2,2,76,"graphics, list, vesa, systems, mail","[graphics, list, vesa, systems, mail, comp, 3d...","[0.03964319722628126, 0.03173609739476984, 0.0...",Topic 2,None,"[96, 293, 340, 18, 552]",None
3,3,74,"pain, doctor, said, surgery, rtrace","[pain, doctor, said, surgery, rtrace, group, t...","[0.03799421560428317, 0.020922120511162413, 0....",Topic 3,None,"[394, 147, 407, 458, 174]",None
4,4,73,"car, cars, engine, std, road","[car, cars, engine, std, road, oil, ford, audi...","[0.06352874584225345, 0.025138172659216432, 0....",Topic 4,None,"[2, 60, 556, 4, 324]",None
5,5,69,"jb, __, sky, don, hst","[jb, __, sky, don, hst, think, shuttle, right,...","[0.0438060931507716, 0.01948286094746268, 0.01...",Topic 5,None,"[213, 56, 333, 112, 542]",None
6,6,58,"planet, space, spacecraft, solar, earth","[planet, space, spacecraft, solar, earth, venu...","[0.030733795408713502, 0.02929112563399176, 0....",Topic 6,None,"[10, 285, 411, 37, 372]",None
7,7,19,"jpeg, image, gif, images, file","[jpeg, image, gif, images, file, color, format...","[0.15168862306364805, 0.06564001718401413, 0.0...",Topic 7,None,"[36, 69, 487, 142, 551]",None
8,8,5,"tommy mac, ibm cl, msu edu, msu, edu 336","[tommy mac, ibm cl, msu edu, msu, edu 336, 217...","[0.06591158419188063, 0.06591158419188063, 0.0...",Topic 8,None,"[530, 67, 440, 172, 298]",None


---
## 3. Test the adaptation pipeline for free (oracle labeler, no API key)

`OracleLabeler` answers each triplet judgment ("is document A more similar to
B, or to C?") by looking up the true 20 Newsgroups category for each document
snippet — it's a stand-in for a real LLM that is *always right*, used purely
to validate the pipeline mechanically: sampling, prompting, parsing, training,
and refitting, with zero API cost. This is the same approach
`benchmarks/adaptation_quality_report.py` uses for its CI-safe scenario.

If a *perfect* oracle doesn't improve the clustering below, the adaptation
plumbing is broken — not the (real, noisier) LLM.

In [4]:
import json
import re


class OracleLabeler:
    """Perfect triplet-judgment oracle for testing: looks up the true label
    for each shown document snippet via the same truncation the prompt
    builder uses, so no real LLM call is needed to prove the pipeline works.
    """

    _PATTERN = re.compile(r"A: (.*?)\n  B: (.*?)\n  C: (.*?)\n\n", re.DOTALL)

    def __init__(self, documents, labels, n_docs_chars=300):
        self.snippet_to_label = {}
        for doc, lbl in zip(documents, labels):
            snippet = doc[:n_docs_chars] + "..." if len(doc) > n_docs_chars else doc
            self.snippet_to_label[snippet] = int(lbl)
        self.calls = 0

    def call_structured(self, system_prompt, user_prompt, schema, max_tokens=None):
        self.calls += 1
        items = self._PATTERN.findall(user_prompt)
        answers = []
        for a_text, b_text, _c_text in items:
            a_lbl = self.snippet_to_label.get(a_text)
            b_lbl = self.snippet_to_label.get(b_text)
            answers.append("B" if a_lbl is not None and a_lbl == b_lbl else "C")
        return json.dumps({"answers": answers})


oracle = OracleLabeler(documents, true_labels)
print("OracleLabeler ready — answers triplet judgments from ground truth, $0 cost.")

OracleLabeler ready — answers triplet judgments from ground truth, $0 cost.


In [5]:
from tritopic.adaptation import AdaptationConfig, adapt_and_refit

adapt_config = AdaptationConfig(
    adapter_mode="linear",       # pure numpy — no extra dependencies, works everywhere
    n_triplets=500,
    triplet_sampling="entropy",  # focus on the model's least-confident documents
    entropy_top_frac=0.9,
    holdout_frac=0.2,
    linear_epochs=80,
    linear_lr=0.08,
    linear_margin=0.15,
    linear_l2=5e-4,
    cache_path="adaptation_triplet_cache.jsonl",  # repeat runs pay $0 (or 0 oracle calls)
    random_state=42,
    verbose=False,
)

new_model, report = adapt_and_refit(
    model, oracle, config=adapt_config, evaluate=True, labels_true=true_labels
)

print(f"Oracle calls            : {report['n_llm_calls']} "
      f"(cache hits: {report['n_cache_hits']}, unparsed: {report['n_unparsed']})")
print(f"Triplets                : {report['n_train_triplets']} train / "
      f"{report['n_holdout_triplets']} holdout")
print(f"Held-out triplet acc.   : {report['holdout_triplet_acc_before']:.3f} -> "
      f"{report['holdout_triplet_acc_after']:.3f}")
report["comparison"]

/home/nevil/workspace/batch-clustering/tritopic/adaptation/pipeline.py:115: UserWarning: adapt_and_refit: held-out triplet accuracy did not improve (0.712 -> 0.702). The adapted embeddings may not be better for this corpus — check n_triplets, epochs, and LLM judgment quality before trusting them.
  warnings.warn(


Oracle calls            : 0 (cache hits: 500, unparsed: 0)
Triplets                : 396 train / 104 holdout
Held-out triplet acc.   : 0.712 -> 0.702


,variant,n_topics,outlier_ratio,silhouette,fit_seconds,davies_bouldin,calinski_harabasz,stability,coherence_mean,diversity,ari,nmi,cluster_accuracy,holdout_triplet_acc,keyword_overlap_vs_baseline
0,baseline,9.0,0.0,0.019655,0.978934,4.181499,8.050833,0.722943,0.394959,1.00,0.203350,0.270309,0.429329,0.711538,1.000000
1,adapted,10.0,0.0,0.029858,0.955567,4.046352,8.505320,0.709759,0.431871,0.99,0.286699,0.348388,0.487633,0.701923,0.557631


**Reading the table above:** the clustering-level columns (`ari`, `nmi`,
`cluster_accuracy`, `silhouette`, `coherence_mean`, ...) are recomputed from a
*fresh* full fit on all documents, so they're the most reliable signal.
`holdout_triplet_acc` is a stricter, much smaller-sample metric (just the
held-out triplets) and can occasionally lag or dip slightly even when the
full clustering clearly improves — that's noise on a small sample, not a red
flag. Judge the adaptation by the full table, not that one column in
isolation (the same nuance shows up in
`benchmarks/adaptation_quality_report.py`'s real-20NG scenario).

### Inline assertions — proving the pipeline actually works

These are the same checks `tests/test_adaptation_pipeline.py` runs in CI.
Run this cell to confirm the feature works correctly in *your* environment
before trusting it on real data.

In [6]:
# The original model is left untouched — adapt_and_refit always returns a new one.
assert new_model is not model
assert new_model._is_fitted
np.testing.assert_allclose(model.embeddings_, baseline_embeddings)

# The report has everything needed to judge the adaptation.
for key in (
    "mode", "n_llm_calls", "n_cache_hits", "n_unparsed", "n_train_triplets",
    "n_holdout_triplets", "holdout_triplet_acc_before", "holdout_triplet_acc_after",
    "comparison",
):
    assert key in report, f"missing report key: {key}"

assert report["mode"] == "linear"
assert report["n_holdout_triplets"] > 0

df = report["comparison"]
baseline_ari = df.loc[df.variant == "baseline", "ari"].iloc[0]
adapted_ari = df.loc[df.variant == "adapted", "ari"].iloc[0]
print(f"ARI vs ground truth: baseline {baseline_ari:.3f}  ->  adapted {adapted_ari:.3f}")
print("PASS — with a perfect oracle, the adapted embeddings should match or beat baseline ARI.")
assert adapted_ari >= baseline_ari - 0.02, (
    "Adapted ARI regressed more than a small tolerance vs. baseline — "
    "something is wrong with the adaptation pipeline, not the LLM."
)
print("\nAll assertions passed.")

ARI vs ground truth: baseline 0.203  ->  adapted 0.287
PASS — with a perfect oracle, the adapted embeddings should match or beat baseline ARI.

All assertions passed.


---
## 4. Use with a real LLM

Same call, one line different: swap the oracle for a real `LLMLabeler`. Cost
is small — a few hundred triplets at ~8 per batched call, cached to disk so
repeat runs on the same corpus are free. On Claude Haiku 4.5, a full
1000-triplet run costs well under $1.

In [7]:
# ---- Uncomment to run with a real LLM ----

# from tritopic import LLMLabeler
#
# labeler = LLMLabeler(
#     provider="anthropic",
#     api_key="sk-ant-...",
#     model="claude-haiku-4-5",
# )
#
# # In-place refit (mirrors tune_resolution_with_llm's ergonomics):
# model.adapt_embeddings_with_llm(labeler, config=adapt_config)
# print(model.adaptation_diagnostics_["holdout_triplet_acc_before"],
#       "->", model.adaptation_diagnostics_["holdout_triplet_acc_after"])
#
# # Or, to keep the original model untouched and compare side by side:
# # new_model, report = adapt_and_refit(model, labeler, config=adapt_config)

print("Skipped — uncomment the cell above and add a real API key to run this.")

Skipped — uncomment the cell above and add a real API key to run this.


### Real sentence-transformers fine-tuning (optional, needs `pip install "tritopic[adaptation]"`)

For a local embedder, `adapter_mode="finetune"` runs a real 1-epoch,
low-learning-rate sentence-transformers fine-tune (`MultipleNegativesRankingLoss`)
instead of the pure-numpy linear adapter — the full ClusterLLM recipe. It
needs `datasets` and `accelerate`, which the default install doesn't include.

In [8]:
# ---- Uncomment to try real fine-tuning (needs sentence-transformers embeddings + [adaptation] extra) ----

# finetune_config = AdaptationConfig(adapter_mode="finetune", n_triplets=800, epochs=1)
# new_model_ft, report_ft = adapt_and_refit(model, oracle, config=finetune_config, evaluate=True)
# report_ft["comparison"]

print("Skipped — see the comment above. Requires a local embedder and the [adaptation] extra.")

Skipped — see the comment above. Requires a local embedder and the [adaptation] extra.


---
## 5. Cheaper extras (no fine-tuning required)

Two independent levers from Viswanathan et al. (TACL 2024) that work with any
embedder, including API-based ones that can't be fine-tuned at all.

**Keyphrase expansion** asks an LLM for a handful of keyphrases per document
and blends their embedding into the document's own — no fine-tuning needed.
**Low-confidence correction** asks an LLM to re-adjudicate only the documents
the model itself is least sure about (a soft-assignment margin below some
threshold), rather than touching every document.

Both need a real LLM to be useful (a keyphrase list or a topic choice has to
come from *somewhere* semantically meaningful) — shown here as a shape/API
reference rather than executed for free, unlike Sections 2–3 above.

In [9]:
# ---- Uncomment to run with a real LLM ----

# from tritopic import EmbeddingEngine, LLMLabeler
# from tritopic.adaptation import (
#     generate_keyphrases,
#     keyphrase_expand_embeddings,
#     reassign_low_confidence,
# )
#
# labeler = LLMLabeler(provider="anthropic", api_key="sk-ant-...", model="claude-haiku-4-5")
#
# # Keyphrase expansion
# engine = EmbeddingEngine(model_name=model.config.embedding_model, provider=model.config.embedding_provider)
# keyphrases = generate_keyphrases(labeler, documents, n_keyphrases=5, cache_path="keyphrase_cache.jsonl")
# expanded_embeddings = keyphrase_expand_embeddings(documents, keyphrases, engine, weight=0.5)
#
# # Post-hoc low-confidence correction (mutates model.labels_ in place unless dry_run=True)
# corrections = reassign_low_confidence(model, labeler, margin_threshold=0.15, max_docs=200, dry_run=True)
# corrections.head(10)

print("Skipped — uncomment the cell above and add a real API key to run this.")

Skipped — uncomment the cell above and add a real API key to run this.


---
## 6. Summary

| Step | What it does | Needs an API key? |
|---|---|---|
| `adapt_and_refit(model, labeler)` | Adapt the embedder to LLM triplet judgments, return a *new* fitted model | Yes (or an oracle/test labeler, as above) |
| `model.adapt_embeddings_with_llm(labeler)` | Same, but refits *in place* (mirrors `tune_resolution_with_llm`) | Yes |
| `compare_embedders(...)` | Baseline vs. adapted, identical TriTopic config, side by side | No |
| `AdaptationConfig(adapter_mode=...)` | `"linear"` (numpy, any embedder) / `"finetune"` (real ST training) / `"auto"` | — |
| `generate_keyphrases` + `keyphrase_expand_embeddings` | Cheap, no-fine-tune embedding boost | Yes |
| `reassign_low_confidence` | Post-hoc correction of only the least-confident assignments | Yes |

See the [README's *LLM-Guided Embedding Adaptation*](../README.md#llm-guided-embedding-adaptation)
section for the full reference, and `tests/test_adaptation_*.py` /
`benchmarks/adaptation_quality_report.py` for more examples.